In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/soil-grain-size-from-photos/sample_submission.csv
/kaggle/input/competitions/soil-grain-size-from-photos/Training_labels_updated.csv
/kaggle/input/competitions/soil-grain-size-from-photos/ppm_updated.csv
/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_Photos_updated/Training-All_Photos_updated/Samsung_A52_H666_01.jpg
/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_Photos_updated/Training-All_Photos_updated/Motorola_Edge_60_fusion_H374_02.jpg
/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_Photos_updated/Training-All_Photos_updated/Motorola_Edge_60_fusion_H374_03.jpg
/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_Photos_updated/Training-All_Photos_updated/Motorola_Edge_H368_04.jpg
/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_Photos_updated/Training-All_Photos_updated/Samsung_A52_H038_01.jpg
/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
from sklearn.model_selection import KFold

# --- CONFIGURATION ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 1e-4
IMG_SIZE = 384
NUM_WORKERS = 4
SEED = 42

# 11 Fixed Support Points from DIN EN ISO 14688-1
SUPPORT_POINTS = np.array([0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2.0, 6.3, 20.0, 63.0, 200.0])
LOG_SP = np.log10(SUPPORT_POINTS)
# Logarithmic interval weights (length of intervals on log scale)
INTERVAL_WEIGHTS = torch.tensor(LOG_SP[1:] - LOG_SP[:-1], dtype=torch.float32).to(DEVICE)

# --- 1. CUSTOM LOG-WEIGHTED EMD LOSS ---
class LogWeightedEMDLoss(nn.Module):
    def __init__(self, interval_weights):
        super(LogWeightedEMDLoss, self).__init__()
        self.interval_weights = interval_weights

    def forward(self, y_pred, y_true):
        # Input expected: shape (batch_size, 11) -> Cumulative distributions [0, 100]
        # Convert percentages to fractions [0, 1] for stable calculation
        pred_frac = y_pred / 100.0
        true_frac = y_true / 100.0
        
        # Absolute differences at all 11 support points
        diff = torch.abs(pred_frac - true_frac)
        
        # Area estimation per interval using trapezoidal rule or mean of endpoints
        interval_errors = (diff[:, :-1] + diff[:, 1:]) / 2.0
        
        # Logarithmically weighted sum
        emd = torch.sum(interval_errors * self.interval_weights, dim=1)
        
        # Multiply by 500 scaling limit if desired, or keep raw mean for optimization
        return torch.mean(emd) * 500.0

# --- 2. DATASET DEFINITION ---
class SoilDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row['sample_id']) + ".jpg" # Adjust extension if PNG
        img_path = os.path.join(self.img_dir, img_name)
        
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
            
        if self.is_test:
            return image, row['sample_id']
            
        # Target cumulative values
        targets = row[SUPPORT_POINTS.astype(str).tolist()].values.astype(np.float32)
        return image, torch.tensor(targets)

# --- 3. MODEL ARCHITECTURE ---
# Uses torchvision's efficientnet as a feature extractor
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights

class SoilGranulometryModel(nn.Module):
    def __init__(self):
        super(SoilGranulometryModel, self).__init__()
        # Using pre-trained weights tuned for fine textures
        weights = EfficientNet_B4_Weights.DEFAULT
        self.backbone = efficientnet_b4(weights=weights)
        
        # Replace classification head
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        
        # Regression head outputting 10 distinct mass bins
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 10) # 10 intervals between 11 support points
        )

    def forward(self, x):
        features = self.backbone(x)
        # Softmax outputs 10 fractions summing strictly to 1.0
        bin_fractions = torch.softmax(self.head(features), dim=1)
        
        # Prepend a zero column for the initial absolute cumulative point boundary context if starting at 0%
        # However, to explicitly model the 11 points starting from the 0.002mm absolute cumulative value:
        # We assume the cumulative values build up to 100% at the 11th point.
        
        # Compute cumulative sum across the 10 bins
        cumsum = torch.cumsum(bin_fractions, dim=1) * 100.0 # scale to [0, 100]
        
        # The first point is calculated relative to base minimal grain size distributions.
        # To fill 11 outputs where output[-1] is strictly 100:
        # We can append a column of zeros at the front or map explicitly. 
        # Let's cleanly construct the 11 support positions:
        batch_size = x.size(0)
        out = torch.zeros(batch_size, 11).to(x.device)
        out[:, 1:] = cumsum
        # out[:, 0] remains 0 or an estimated initial base fraction. 
        # Alternatively, we predict 11 raw values and normalize/sort, but cumsum ensures perfect validity:
        return out

# --- 4. PREPROCESSING & TRANSFORMS ---
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [3]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

# --- INFERENCE LOOP FOR SUBMISSIONS ---
def generate_submission(model_paths, test_df, test_img_dir, output_path="submission.csv"):
    test_dataset = SoilDataset(test_df, test_img_dir, transform=val_transforms, is_test=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    models = []
    for path in model_paths:
        model = SoilGranulometryModel().to(DEVICE)
        model.load_state_dict(torch.load(path))
        model.eval()
        models.append(model)
        
    predictions = []
    sample_ids = []
    
    with torch.no_grad():
        for images, ids in test_loader:
            images = images.to(DEVICE)
            # Ensemble predictions across multiple folds
            fold_preds = torch.zeros(images.size(0), 11).to(DEVICE)
            for model in models:
                fold_preds += model(images)
            fold_preds /= len(models)
            
            # Ensure the final column is mathematically capped at exactly 100.00
            fold_preds[:, -1] = 100.00
            
            predictions.append(fold_preds.cpu().numpy())
            sample_ids.extend(ids)
            
    pred_array = np.vstack(predictions)
    col_names = ['0.002', '0.0063', '0.02', '0.063', '0.2', '0.63', '2.0', '6.3', '20.0', '63.0', '200.0']
    
    sub_df = pd.DataFrame(pred_array, columns=col_names)
    sub_df.insert(0, 'sample_id', sample_ids)
    
    # Final sanity check: force monotonically increasing sequences
    for i in range(1, len(col_names)):
        sub_df[col_names[i]] = np.maximum(sub_df[col_names[i]], sub_df[col_names[i-1]])
    sub_df[col_names[-1]] = 100.00

    sub_df.to_csv(output_path, index=False)
    print(f"Submission successfully saved to {output_path}!")